# Comparación local con SRBench 2025 en Google Colab

Ejecuta el perfil `official` del **harness local de WarpSymbolic**: 24 datasets × 30 semillas = **720 tareas**. Esto sirve para diagnóstico interno y una comparación exploratoria con resultados publicados. **No ejecuta `experiment/analyze.py` de SRBench, no reproduce su ajuste de hiperparámetros ni constituye una evaluación aceptada por los creadores.** El JSONL declara `official_protocol=false`.

Antes de comenzar, selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. El perfil fija **3600 s como techo por tarea**, 50.000 individuos y 150 generaciones. El runner y la búsqueda adaptativa respetan ahora el techo de tiempo; una tarea puede terminar antes al alcanzar otro criterio de parada. Colab puede interrumpir una sesión; vuelve a ejecutar las celdas desde el principio para reanudar. Cada tarea terminada queda guardada en Google Drive.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/WarpSymbolic/SRBench2025_cuda_fix_1')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
REF_FILE = RUN_ROOT / 'commit.txt'
REPO = Path('/content/WarpSymbolic_cuda_fix_1')
REPO_URL = 'https://github.com/juansito17/Algoritmo-Genetico---Formulas.git'
GIT_REF = 'codex/fix-srbench-lexicase-cuda'
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', GIT_REF, REPO_URL, str(REPO)], check=True)
if REF_FILE.exists():
    pinned_commit = REF_FILE.read_text(encoding='utf-8').strip()
    current_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    if current_commit != pinned_commit:
        subprocess.run(['git', 'fetch', '--depth', '1', 'origin', pinned_commit], cwd=REPO, check=True)
        subprocess.run(['git', 'checkout', '--detach', pinned_commit], cwd=REPO, check=True)
else:
    pinned_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    REF_FILE.write_text(pinned_commit + '\n', encoding='utf-8')
OUTPUT = RUN_ROOT / pinned_commit[:12] / 'official.jsonl'
CACHE = RUN_ROOT / 'cache'
RANKING = OUTPUT.with_name('ranking.json')
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print('Commit fijado:', pinned_commit)
print('Resultados:', OUTPUT)

## Instalar y verificar CUDA

La compilación nativa tarda varios minutos al iniciar una sesión nueva de Colab. Se detiene si no hay GPU CUDA o si la extensión no carga.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO) + '[benchmark]'], check=True)
# pip se ejecuta en un subproceso; el kernel actual puede no actualizar sys.path.
src_dir = str(REPO / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
runner_source = (REPO / 'src/warpsymbolic/cli/srbench_runner.py').read_text(encoding='utf-8')
adaptive_source = (REPO / 'src/AlphaSymbolic/experimental/adaptive_search.py').read_text(encoding='utf-8')
cuda_source = (REPO / 'src/warpsymbolic/gpu/cuda/rpn_kernels.cu').read_text(encoding='utf-8')
if 'const int error_cases = abs_errors.size(1)' not in cuda_source:
    raise RuntimeError('El commit fijado no incluye la corrección del acceso fuera de límites en lexicase. Publícala antes de ejecutar.')
if 'min(float(context.fit_time_limit_sec), 60.0)' in runner_source or 'min(float(estimator.max_time), 60.0)' in adaptive_source:
    raise RuntimeError('El commit fijado aún recorta la búsqueda a 60 s. Publica la corrección y usa un RUN_ROOT nuevo antes de medir.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Activa una GPU CUDA en la configuración de Colab.')
device = torch.cuda.get_device_properties(0)
print(f'GPU: {device.name}; VRAM: {device.total_memory / 2**30:.1f} GiB; PyTorch: {torch.__version__}')
build_env = os.environ.copy()
build_env['TORCH_CUDA_ARCH_LIST'] = f'{device.major}.{device.minor}'
build_env['MAX_JOBS'] = '2'
cuda_dir = REPO / 'src' / 'warpsymbolic' / 'gpu' / 'cuda'
extensions = list(cuda_dir.glob('rpn_cuda_native*.so'))
sources = [path for pattern in ('*.cu', '*.cpp', '*.h') for path in cuda_dir.glob(pattern)]
if not extensions or max(path.stat().st_mtime for path in sources) > max(path.stat().st_mtime for path in extensions):
    subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace'], cwd=cuda_dir, env=build_env, check=True)
from warpsymbolic.gpu.cuda_loader import load_rpn_cuda_native
print('Extensión CUDA:', load_rpn_cuda_native().__file__)
subprocess.run(['nvidia-smi'], check=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
environment_file = OUTPUT.parent / f'environment_{stamp}.txt'
with environment_file.open('w', encoding='utf-8') as record:
    record.write(f'commit={pinned_commit}\npython={sys.version}\ntorch={torch.__version__}\ntorch_cuda={torch.version.cuda}\n')
    for args in (['nvidia-smi'], ['nvcc', '--version'], [sys.executable, '-m', 'pip', 'freeze']):
        record.write('\n$ ' + ' '.join(args) + '\n')
        record.write(subprocess.run(args, capture_output=True, text=True, check=True).stdout)
print('Entorno guardado:', environment_file)

## Verificar el plan

Esta celda no entrena: confirma la cobertura y los presupuestos configurados. Verifica que el commit clonado de WarpSymbolic incluya la eliminación del antiguo recorte interno de 60 s.

In [ ]:
BASE = [sys.executable, '-u', '-m', 'warpsymbolic.cli.benchmark_srbench', '--profile', 'official', '--cache-dir', str(CACHE), '--output', str(OUTPUT), '--resume']
subprocess.run(BASE + ['--dry-run'], cwd=REPO, check=True)

## Preparar datos y referencias

Descarga y comprueba hashes de los 24 datasets y de los resultados oficiales usados para el ranking. El caché queda en Drive para las sesiones siguientes.

In [ ]:
subprocess.run(BASE + ['--prepare-only', '--rank'], cwd=REPO, check=True)

## Prueba GPU antes de las 720 tareas

Ejecuta una sola tarea que fallaba con el kernel anterior. La celda detiene el proceso si el motor CUDA vuelve a registrar un error. Guarda la prueba por separado en Drive y no la mezcla con el benchmark.

In [ ]:
from warpsymbolic.cli.benchmark_srbench import read_jsonl
preflight_output = OUTPUT.parent / 'cuda_preflight.jsonl'
preflight_command = BASE + ['--datasets', '1193_BNG_lowbwt', '--seeds', '23654', '--output', str(preflight_output)]
subprocess.run(preflight_command, cwd=REPO, check=True)
preflight_rows = list(read_jsonl(preflight_output))
if len(preflight_rows) != 1 or preflight_rows[0].get('status') != 'ok':
    raise RuntimeError(f'Prueba GPU fallida: {preflight_rows[-1].get("error") if preflight_rows else "sin registro"}')
preflight_metadata = preflight_rows[0].get('runner_metadata') or {}
if preflight_metadata.get('engine_error'):
    raise RuntimeError(f'Error interno GPU: {preflight_metadata["engine_error"]}')
print('Prueba GPU correcta; ganador:', preflight_metadata.get('winner'))


## Ejecutar las 720 tareas de la comparación local

La salida aparece en vivo por dataset y semilla. Los registros se escriben y sincronizan en Drive al terminar cada tarea. Si Colab corta la sesión, repite las celdas: `--resume` salta los registros ya guardados. Los errores previos también se saltan; para reintentarlos añade `--retry-failed` al comando.

In [ ]:
import queue, threading, time
from warpsymbolic.cli.benchmark_srbench import read_jsonl
run_env = os.environ.copy()
run_env['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(BASE, cwd=REPO, env=run_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
lines = queue.Queue()
def forward_output():
    for line in process.stdout:
        lines.put(line)
    lines.put(None)
threading.Thread(target=forward_output, daemon=True).start()
started = time.monotonic()
print('Benchmark iniciado; la salida y el estado se actualizarán en vivo.', flush=True)
while True:
    try:
        line = lines.get(timeout=30)
    except queue.Empty:
        saved = sum(1 for row in read_jsonl(OUTPUT) if row.get('record_type') == 'srbench_run') if OUTPUT.exists() else 0
        print(f'En curso: {time.monotonic() - started:.0f} s; registros guardados: {saved}/720', flush=True)
        continue
    if line is None:
        break
    print(line, end='', flush=True)
returncode = process.wait()
print('Código de salida:', returncode, '| JSONL:', OUTPUT, flush=True)
if returncode:
    print('Revisa los registros con status=error antes de generar el ranking final.')

## Comprobar cobertura y comparar

Genera el ranking solo cuando los 720 pares dataset/semilla tengan un registro exitoso. Muestra automáticamente el puesto estimado de WarpSymbolic, sus métricas principales, los diez primeros métodos de referencia y una auditoría del motor evolutivo. `status=ok` confirma una predicción válida, pero puede incluir una fórmula alternativa si el motor GPU falló. El archivo `ranking.json` queda junto al JSONL en Drive.

In [ ]:
from warpsymbolic.cli.benchmark_srbench import load_manifest, read_jsonl, DEFAULT_MANIFEST
manifest = load_manifest(DEFAULT_MANIFEST)
expected = {(item['name'], int(seed)) for item in manifest['datasets'] for seed in manifest['seeds']}
latest = {}
for row in read_jsonl(OUTPUT):
    if row.get('record_type') == 'srbench_run':
        latest[(row['dataset'], int(row['seed']))] = row
successful = {key for key, row in latest.items() if row.get('status') == 'ok'}
failed = {key for key, row in latest.items() if row.get('status') != 'ok'}
print(f'Completadas: {len(successful & expected)}/{len(expected)}; errores: {len(failed & expected)}; pendientes: {len(expected - successful - failed)}')
from collections import Counter
ok_rows = [latest[key] for key in successful & expected]
winners = Counter((row.get('runner_metadata') or {}).get('winner') for row in ok_rows)
gpu_errors = [(row['dataset'], row['seed'], (row.get('runner_metadata') or {}).get('engine_error')) for row in ok_rows if (row.get('runner_metadata') or {}).get('engine_error')]
engine_winners = sum(count for name, count in winners.items() if isinstance(name, str) and name.startswith('engine_'))
times = sorted(float(row['training_time_sec']) for row in ok_rows)
print(f'Motor evolutivo ganador: {engine_winners}/{len(ok_rows)}; errores internos GPU: {len(gpu_errors)}')
print('Ganadores más frecuentes:', winners.most_common(8))
if times:
    print(f'Tiempo de entrenamiento: mínimo {times[0]:.2f}s; mediana {times[len(times)//2]:.2f}s; máximo {times[-1]:.2f}s')
if gpu_errors:
    print('Primeros errores GPU:', gpu_errors[:3])
    print('ATENCIÓN: revisa estos errores antes de interpretar el puesto estimado como rendimiento del motor evolutivo.')
if successful == expected and not failed and not gpu_errors:
    subprocess.run(BASE + ['--rank-only', '--ranking-output', str(RANKING)], cwd=REPO, check=True)
    ranking = json.loads(RANKING.read_text(encoding='utf-8'))
    print('Ranking:', RANKING)
    local_rows = [row for row in ranking['algorithm_ranking'] if row['algorithm'] == 'WarpSymbolic [local]']
    if len(local_rows) != 1:
        raise RuntimeError(f'Se esperaba una fila WarpSymbolic [local]; encontradas: {len(local_rows)}')
    local = local_rows[0]
    print(f"Puesto estimado: {local['estimated_reference_r2_rank_position']} de {len(ranking['algorithm_ranking'])} métodos comparados")
    print(f"Rango R² medio: {local['mean_r2_rank']:.2f}; R² mediano: {local['median_r2_test']:.3f}; datasets: {local['datasets_present']}")
    print('Primeros diez por rango R² medio:')
    for row in ranking['algorithm_ranking'][:10]:
        print(f"  {row['estimated_reference_r2_rank_position']:>2}. {row['algorithm']}: {row['mean_r2_rank']:.2f}")
    print('Comparable con protocolo oficial:', ranking['comparable_to_official'])
    if not ranking['comparable_to_official']:
        print('Este puesto es una comparación local con referencias publicadas; no es una posición oficial de SRBench.')
elif gpu_errors:
    print('Ranking bloqueado: el motor GPU falló en ejecuciones guardadas como ok. El ranking.json anterior incluye fallbacks y no mide el motor evolutivo.')
    print('No repitas las 720 tareas todavía: primero diagnostica CUDA en una sola tarea y usa un RUN_ROOT nuevo tras corregir el fallo.')
else:
    print('Ejecuta de nuevo la celda de benchmark para continuar; usa --retry-failed si hubo errores.')

## Diagnóstico CUDA si aparecen errores internos

Ejecuta esta celda solo si la auditoría informa errores GPU. Reproduce una tarea en un proceso independiente con `CUDA_LAUNCH_BLOCKING=1` y el modo de excepción inmediata, y guarda el registro en un archivo nuevo; no modifica los 720 resultados. Si el commit clonado aún no contiene el modo de diagnóstico, esta celda modifica solo su copia temporal en Colab. Comparte la traza completa para localizar la operación CUDA.

In [ ]:
from warpsymbolic.cli.benchmark_srbench import read_jsonl
debug_source_path = REPO / 'src/AlphaSymbolic/experimental/adaptive_search.py'
debug_source = debug_source_path.read_text(encoding='utf-8')
if 'WARPSYMBOLIC_RAISE_ENGINE_ERROR' not in debug_source:
    needle = '            except Exception as exc:\n                posterior[arm_name][1] += 1.0'
    if debug_source.count(needle) != 2:
        raise RuntimeError('No se encontró el punto de captura esperado; no se modificó el código')
    debug_source_path.write_text(debug_source.replace(needle, '            except Exception as exc:\n                raise\n                posterior[arm_name][1] += 1.0'), encoding='utf-8')
    print('Copia temporal preparada para mostrar la traza CUDA completa')
debug_output = OUTPUT.parent / f'cuda_debug_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.jsonl'
debug_env = os.environ.copy()
debug_env['CUDA_LAUNCH_BLOCKING'] = '1'
debug_env['WARPSYMBOLIC_RAISE_ENGINE_ERROR'] = '1'
debug_env['PYTHONUNBUFFERED'] = '1'
debug_command = BASE + ['--datasets', '1193_BNG_lowbwt', '--seeds', '23654', '--output', str(debug_output)]
print('Registro de diagnóstico:', debug_output, flush=True)
debug_process = subprocess.run(debug_command, cwd=REPO, env=debug_env)
print('Código de salida:', debug_process.returncode)
debug_rows = list(read_jsonl(debug_output)) if debug_output.exists() else []
if debug_rows:
    print('Estado:', debug_rows[-1].get('status'))
    print('Traza de error:', debug_rows[-1].get('error_traceback'))


## Antes de solicitar una evaluación de SRBench

Esta comparación local no reemplaza las pruebas de los creadores. El repositorio contiene `integrations/srbench/prepare_upstream.sh` para copiar el método a un checkout fijado de SRBench y `integrations/srbench/run_upstream_24x30.sh` para usar `experiment/analyze.py`. Antes de pedir revisión hay que publicar y fijar un commit de WarpSymbolic, construir y probar su imagen en un entorno Linux con CUDA, ejecutar los tests upstream y conservar sus resultados brutos y el registro de hardware. La aceptación depende de la revisión de los mantenedores.